<a href="https://colab.research.google.com/github/dpthinh10dona/StrongPasswordGenerator/blob/main/StrengthPasswordGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os

# 1. SETUP KAGGLE & DOWNLOAD DATASET
os.environ['KAGGLE_USERNAME'] = "dpthinh10dona"
os.environ['KAGGLE_KEY'] = "da che" # Thay bằng key thật của bạn

!pip install -q torch pandas scikit-learn
!kaggle datasets download -d jeffersonvalandro/password-dataset
!unzip -o password-dataset.zip

# ============================================
# FULL PIPELINE: Password AI System (Colab Ready)
# ============================================

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# Thiết lập thiết bị (GPU nếu có)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Đang chạy trên thiết bị: {device.upper()}")

# ============================================
# 2. LOAD & CLEAN DATA (CẬP NHẬT THEO ẢNH THỰC TẾ)
# ============================================
print("⏳ Đang tải và làm sạch dữ liệu...")
df = pd.read_csv('passwords_dataset.csv')

df = df.dropna(subset=['Password', 'Strength'])
df['Password'] = df['Password'].astype(str)

# THÊM BƯỚC NÀY: Ánh xạ (Map) chữ sang số học
strength_mapping = {
    'Weak': 0,
    'Other': 1,
    'Strong': 2
}

# Thay thế các chữ bằng số tương ứng. Nếu gặp chữ lạ không có trong từ điển thì mặc định cho là 0 (Weak)
df['Strength'] = df['Strength'].map(strength_mapping).fillna(0).astype(int)

# Lấy mẫu để chạy nhanh trên Colab
df_rf = df.sample(50000) if len(df) > 50000 else df
df_tf = df.head(10000)

# ============================================
# 3. TRAIN STRENGTH CLASSIFIER (Random Forest)
# ============================================
def extract_features(pw):
    pw = str(pw)
    return [
        len(pw),
        sum(c.islower() for c in pw),
        sum(c.isupper() for c in pw),
        sum(c.isdigit() for c in pw),
        sum(c in "!@#$%^&*" for c in pw)
    ]

# Đã đổi thành 'Password' (P viết hoa)
X = [extract_features(p) for p in df_rf['Password']]
y = df_rf['Strength']

clf = RandomForestClassifier(n_estimators=50, n_jobs=-1)
clf.fit(X, y)
print("✅ Mô hình đánh giá độ mạnh (Classifier) đã train xong!")

# ============================================
# 4. PREPARE TRANSFORMER DATASET
# ============================================
def extract_flags(pw):
    return {
        "LOWER": int(any(c.islower() for c in pw)),
        "UPPER": int(any(c.isupper() for c in pw)),
        "NUM": int(any(c.isdigit() for c in pw)),
        "SPEC": int(any(c in "!@#$%^&*" for c in pw)),
    }

lines = []
# Đã đổi thành 'Password' (P viết hoa)
for pw in df_tf['Password']:
    f = extract_flags(pw)
    line = f"<LOWER={f['LOWER']}><UPPER={f['UPPER']}><NUM={f['NUM']}><SPEC={f['SPEC']}>:{pw}"
    lines.append(line)

all_text = ''.join(lines)
chars = sorted(list(set(all_text)))
char2idx = {c: i+1 for i, c in enumerate(chars)}
char2idx['<PAD>'] = 0
idx2char = {i: c for c, i in char2idx.items()}

vocab_size = len(char2idx)
max_len = 50

class PasswordDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def encode(self, text):
        ids = [char2idx.get(c, 0) for c in text]
        ids = ids[:max_len]
        ids += [0] * (max_len - len(ids))
        return torch.tensor(ids)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        x = self.encode(text[:-1])
        y = self.encode(text[1:])
        return x, y

dataset = PasswordDataset(lines)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# ============================================
# 5. TRANSFORMER MODEL
# ============================================
class MiniTransformer(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        d_model = 128
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Parameter(torch.randn(1, 100, d_model))

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=4, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        x = self.embedding(x) + self.pos_embedding[:, :seq_len, :]
        x = self.transformer(x)
        return self.fc(x)

# ============================================
# 6. TRAIN GENERATOR (CÓ THANH TIẾN TRÌNH ETA)
# ============================================
from tqdm.auto import tqdm # Thêm thư viện này
import time

print("⏳ Đang huấn luyện AI tạo mật khẩu (Transformer)...")
model = MiniTransformer(vocab_size).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=0)

EPOCHS = 5

# Bắt đầu đo tổng thời gian
start_time = time.time()

for epoch in range(EPOCHS):
    total_loss = 0

    # 🌟 BỌC DATALOADER VÀO TQDM ĐỂ TẠO THANH TIẾN TRÌNH 🌟
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)

    for x, y in progress_bar:
        x, y = x.to(device), y.to(device)

        out = model(x)
        out = out.reshape(-1, vocab_size)
        y = y.reshape(-1)

        loss = criterion(out, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Cập nhật loss hiển thị trực tiếp trên thanh tiến trình
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    print(f"👉 Tổng kết Epoch {epoch+1}: Loss trung bình = {total_loss/len(dataloader):.4f}")

# Tính tổng thời gian đã chạy
end_time = time.time()
total_mins = (end_time - start_time) / 60
print(f"✅ Mô hình Transformer đã train xong! Tổng thời gian: {total_mins:.2f} phút.")

# ============================================
# 7. GENERATOR & PIPELINE FUNCTIONS (CÓ LUẬT THÉP KIỂM DUYỆT)
# ============================================
def generate(model, start_text, max_gen=30, temperature=0.8):
    model.eval()
    input_ids = [char2idx.get(c, 0) for c in start_text]

    for _ in range(max_gen):
        x_input = torch.tensor([input_ids]).to(device)
        with torch.no_grad():
            out = model(x_input)
        probs = torch.softmax(out[0, -1] / temperature, dim=0).cpu()
        next_id = torch.multinomial(probs, 1).item()

        if next_id == 0:
            break
        input_ids.append(next_id)

    result = ''.join([idx2char.get(i, '') for i in input_ids])
    if ":" in result:
        return result.split(":")[-1].strip()
    return result

def upgrade_weak_password(weak_pw, criteria, min_strength=2, max_attempts=100, max_add_chars=6):
    prompt = f"<LOWER={criteria['LOWER']}><UPPER={criteria['UPPER']}><NUM={criteria['NUM']}><SPEC={criteria['SPEC']}>:{weak_pw}"

    # Ép AI thử tối đa 100 lần để tìm ra chuỗi hoàn hảo
    for _ in range(max_attempts):
        enhanced_pw = generate(model, prompt, max_gen=max_add_chars)

        if not enhanced_pw or enhanced_pw == weak_pw:
            continue

        # 1. Cho AI chấm điểm
        score = clf.predict([extract_features(enhanced_pw)])[0]

        # 2. KIỂM DUYỆT LUẬT THÉP: Bắt buộc đếm bằng code Python
        has_lower = any(c.islower() for c in enhanced_pw)
        has_upper = any(c.isupper() for c in enhanced_pw)
        has_num = any(c.isdigit() for c in enhanced_pw)
        has_spec = any(c in "!@#$%^&*" for c in enhanced_pw)

        # Mặc định cho là đỗ, trừ khi vi phạm tiêu chí
        is_valid = True
        if criteria.get("LOWER") == 1 and not has_lower: is_valid = False
        if criteria.get("UPPER") == 1 and not has_upper: is_valid = False
        if criteria.get("NUM") == 1 and not has_num: is_valid = False
        if criteria.get("SPEC") == 1 and not has_spec: is_valid = False

        # Chỉ trả về kết quả khi VỪA điểm cao, VỪA đủ tiêu chí cứng
        if score >= min_strength and is_valid:
            return enhanced_pw, score

    # Nếu xui quá chạy 100 lần không ra, trả về cái cuối cùng nó nghĩ ra
    return enhanced_pw, score

# ============================================
# 8. TEST SYSTEM
# ============================================
print("\n🔥 HỆ THỐNG NÂNG CẤP MẬT KHẨU BẰNG AI (BẢN STRICT) 🔥")

weak_password = input("👉 Nhập mật khẩu yếu (vd: tên bạn, ngày sinh...): ")

criteria = {
    "LOWER": 1,
    "UPPER": 1,
    "NUM": 0,
    "SPEC": 1
}

print("⏳ Đang ép AI 'xào nấu' đúng tiêu chí...")

# Mẹo: Tăng max_add_chars lên 6 hoặc 8 để AI có đủ "chỗ trống" chèn Ký tự đặc biệt và Số vào!
enhanced_pw, score = upgrade_weak_password(
    weak_password,
    criteria,
    min_strength=2,
    max_attempts=150,   # Cho nó thử sái cổ 150 lần
    max_add_chars=4     # Cho phép đắp thêm tối đa 6 ký tự
)

print("\n" + "="*40)
print(f"❌ Mật khẩu ban đầu : {weak_password}")
print(f"✅ Mật khẩu nâng cấp: {enhanced_pw}")
print(f"📊 Điểm độ mạnh    : {score} (0: Yếu, 1: Vừa, 2: Mạnh)")
print("="*40)

Dataset URL: https://www.kaggle.com/datasets/jeffersonvalandro/password-dataset
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
password-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  password-dataset.zip
  inflating: passwords_dataset.csv   
🚀 Đang chạy trên thiết bị: CUDA
⏳ Đang tải và làm sạch dữ liệu...
✅ Mô hình đánh giá độ mạnh (Classifier) đã train xong!
⏳ Đang huấn luyện AI tạo mật khẩu (Transformer)...


Epoch 1/5:   0%|          | 0/157 [00:00<?, ?it/s]

👉 Tổng kết Epoch 1: Loss trung bình = 1.1203


Epoch 2/5:   0%|          | 0/157 [00:00<?, ?it/s]

👉 Tổng kết Epoch 2: Loss trung bình = 0.2902


...

👉 Tổng kết Epoch 3: Loss trung bình = 0.1135


Epoch 4/5:   0%|          | 0/157 [00:00<?, ?it/s]

👉 Tổng kết Epoch 4: Loss trung bình = 0.1107


Epoch 5/5:   0%|          | 0/157 [00:00<?, ?it/s]

👉 Tổng kết Epoch 5: Loss trung bình = 0.1096
✅ Mô hình Transformer đã train xong! Tổng thời gian: 0.22 phút.

🔥 HỆ THỐNG NÂNG CẤP MẬT KHẨU BẰNG AI (BẢN STRICT) 🔥
👉 Nhập mật khẩu yếu (vd: tên bạn, ngày sinh...): chubbydthw
⏳ Đang ép AI 'xào nấu' đúng tiêu chí...

❌ Mật khẩu ban đầu : chubbydthw
✅ Mật khẩu nâng cấp: chubbydthwu@LL
📊 Điểm độ mạnh    : 2 (0: Yếu, 1: Vừa, 2: Mạnh)
